In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

model_id = 'HuggingFaceTB/SmolLM2-135M'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).cuda()

streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=False, skip_special_tokens=False)
prompt = """\
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Suggest 2 ways to lose my weight.

Answer:"""

_ = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(model.device),
    streamer=streamer,
    max_length=100,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
)
# print(tokenizer.decode(_[0]))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-135M-Instruct')
model = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM2-135M-Instruct').cuda()
streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=False, skip_special_tokens=False)
def generate(messages):

    # continue_final_message=True
    chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_input, return_tensors="pt").to(model.device)

    _ = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.6,
        do_sample=True,
        streamer=streamer
    )
generate([
        {"role": "user", "content": "What are the three primary colors?"},
])


In [ ]:
import torch
prompt = "I want some French"

# tokenize
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# logits for the NEXT token
next_token_logits = logits[0, -1]

# probabilities
probs = torch.softmax(next_token_logits, dim=-1)

# top 20
topk = torch.topk(probs, k=20)

ids = topk.indices.tolist()
probs = topk.values.tolist()

for id, p in zip(ids, probs):
    print(f"{tokenizer.decode(id)}  {p:.4f}")